# L5 · Cómo entrena LeRobot

`lerobot-train` es un comando que, con las opciones correctas, deja un modelo
entrenado en una carpeta. Este notebook lo abre: armamos a mano el mismo ciclo
de entrenamiento, lo corremos unos minutos en la Mac, y después usamos el
comando de verdad para ver que hace lo mismo.

- **Lo principal (1–4):** qué arma el entrenamiento, el ciclo paso a paso, la
  prueba de cordura de "memorizar poco", y el comando real con sus resultados.
- **Para profundizar (5–6):** cuánto cuesta entrenar en serio y qué perillas
  importan.

Usa el dataset chiquito de 3 episodios del notebook 02 (`nb_demo`).

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from lerobot.configs.types import FeatureType
from lerobot.datasets.factory import IMAGENET_STATS, resolve_delta_timestamps
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors

from toolkit.paths import DATA_DIR, DATASETS_DIR

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"   # la GPU de la Mac, si está
DEMO_ROOT = DATASETS_DIR / "local" / "nb_demo"
CAMERA = "observation.images.cam_right_high"
print("dispositivo:", DEVICE)

## 1. Qué es "entrenar" acá

Es aprendizaje supervisado: mostrarle a la red muchos ejemplos de "esto veías →
estas 48 acciones hizo el operador" y ajustar sus pesos para que su respuesta
se parezca cada vez más.

Tres diferencias con lo que quizás tenés en la cabeza de otros proyectos de ML:

- **Se cuenta en pasos, no en épocas.** Un paso es: tomar un lote al azar,
  calcular la pérdida, ajustar los pesos. `--steps=30000` son 30.000 de esos.
  Cuántas veces se recorre el dataset entero es una consecuencia.
- **Cada ejemplo es un cuadro**, y cuadros vecinos son casi idénticos. 22.000
  cuadros no son 22.000 ejemplos independientes: son 61 demostraciones.
- **No hay un número de validación que decida.** La pérdida en datos no vistos
  correlaciona mal con el éxito en el robot (robomimic: el mejor checkpoint por
  validación rinde 50 a 100% peor que el mejor real). Se elige mirando el
  robot o la simulación.

## 2. Las piezas que arma `lerobot-train`

En orden, lo mismo que hace el comando por dentro:

### 2.1 La configuración de la política

Partimos de la de nuestra primera ACT (versión liviana). Las *features* (qué
entra y qué sale) salen de la metadata del dataset.

In [ ]:
meta = LeRobotDatasetMetadata("local/nb_demo", root=DEMO_ROOT)
features = dataset_to_policy_features(meta.features)
outputs = {k: v for k, v in features.items() if v.type is FeatureType.ACTION}
inputs = {k: v for k, v in features.items() if k not in outputs}

config = ACTConfig(
    input_features=inputs, output_features=outputs, device=DEVICE,
    chunk_size=48, n_action_steps=48, dim_model=192, dim_feedforward=768, n_heads=4,
)
print("entradas:", list(inputs), "\nsalidas:", list(outputs))

### 2.2 El dataset, con la ventana que pide la política

La configuración de ACT dice "48 acciones futuras"; `resolve_delta_timestamps`
lo traduce a segundos (L1, L2).

In [ ]:
delta_timestamps = resolve_delta_timestamps(config, meta)
dataset = LeRobotDataset("local/nb_demo", root=DEMO_ROOT, video_backend="pyav", delta_timestamps=delta_timestamps)
print(len(dataset), "ejemplos (cuadros) ·", {k: len(v) for k, v in delta_timestamps.items()})

### 2.3 Los procesadores, con las estadísticas del dataset

Acá aparece un detalle que `lerobot-train` hace por defecto
(`use_imagenet_stats=true`): **reemplaza las estadísticas de la cámara por las
de ImageNet** antes de crear el normalizador, para hablarle al ResNet18 en su
idioma (L4). Lo hacemos igual.

In [ ]:
stats = {k: dict(v) for k, v in meta.stats.items()}
for camera in meta.camera_keys:
    for stat, value in IMAGENET_STATS.items():
        stats[camera][stat] = torch.tensor(value, dtype=torch.float32)
preprocessor, postprocessor = make_pre_post_processors(config, dataset_stats=stats)
print("normalización de la cámara:", stats[CAMERA]["mean"].ravel().tolist())

### 2.4 La política y el optimizador

ACT trae su optimizador "de fábrica": **AdamW** con **dos tasas de
aprendizaje**, una para el ResNet (que ya sabe ver, conviene tocarlo poco) y
otra para el resto (que arranca de cero).

In [ ]:
policy = ACTPolicy(config).to(DEVICE)
groups = policy.get_optim_params()
optimizer = config.get_optimizer_preset().build(groups)
backbone_params = sum(p.numel() for p in groups[1]["params"])
rest_params = sum(p.numel() for p in groups[0]["params"])
print(f"resto de la red : {rest_params / 1e6:.1f} M parámetros, lr {optimizer.param_groups[0]['lr']}")
print(f"backbone ResNet : {backbone_params / 1e6:.1f} M parámetros, lr {optimizer.param_groups[1]['lr']}")

### 2.5 El cargador de lotes

Un `DataLoader` común de PyTorch que baraja los cuadros: cada lote mezcla
momentos de episodios distintos.

In [ ]:
loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0, drop_last=True)

## 3. El ciclo, paso a paso

Esto es lo que hace `update_policy` en `lerobot/scripts/lerobot_train.py`:

1. Tomar un lote y pasarlo por el **preprocesador** (normalizar).
2. **forward**: la política calcula la pérdida (L1 + KL, L3).
3. **backward**: calcular cuánto contribuye cada peso al error (gradientes).
4. **Recortar gradientes** a norma 10: si un lote raro produce gradientes
   enormes, se achican para que un solo paso no rompa lo aprendido.
5. **Paso del optimizador**: mover los pesos.

Corremos 300 pasos (un par de minutos en la Mac):

In [ ]:
history = {"l1": [], "kl": [], "grad_norm": []}
batches = iter(loader)
start = time.time()
for step in range(300):
    try:
        batch = next(batches)
    except StopIteration:
        batches = iter(loader)
        batch = next(batches)

    batch = preprocessor(batch)
    policy.train()
    loss, parts = policy.forward(batch)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
    optimizer.step()
    optimizer.zero_grad()

    history["l1"].append(parts["l1_loss"])
    history["kl"].append(parts["kld_loss"])
    history["grad_norm"].append(float(grad_norm))
seconds_per_step = (time.time() - start) / 300
print(f"{seconds_per_step:.2f} s por paso con lotes de 8 en {DEVICE}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, (name, values) in zip(axes, history.items()):
    ax.plot(values, alpha=0.4)
    ax.plot(np.convolve(values, np.ones(20) / 20, mode="valid"), lw=2)
    ax.set_title(name); ax.set_xlabel("paso")
axes[1].set_yscale("log")
plt.tight_layout()

- **L1** baja: la red se acerca a las acciones del operador.
- **KL** arranca alto (el encoder del VAE produce `z` arbitrarios) y cae rápido
  hacia cero: es el empuje que, sostenido 30.000 pasos con `kl_weight=10`,
  termina en el *posterior collapse* que vimos en el L3.
- **La norma del gradiente** arranca muy por encima de 10 (se está recortando
  en cada paso) y baja a medida que la red se acomoda.

## 4. La prueba de cordura: ¿puede memorizar?

Antes de entrenar en serio, la prueba clásica: **con muy pocos datos, la red
tiene que poder memorizarlos.** Si con 3 episodios la pérdida no baja, hay un
bug (datos mal alineados, normalización rota, tasa de aprendizaje absurda), y
entrenar 10 horas con todo el dataset no lo va a arreglar.

Seguimos un rato más y miramos cómo predice sobre un episodio que **sí** vio:

In [ ]:
for step in range(300):
    try:
        batch = next(batches)
    except StopIteration:
        batches = iter(loader)
        batch = next(batches)
    loss, parts = policy.forward(preprocessor(batch))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
    optimizer.step(); optimizer.zero_grad()
    history["l1"].append(parts["l1_loss"])
print(f"L1 tras {len(history['l1'])} pasos: {np.mean(history['l1'][-20:]):.3f} (arrancó en {np.mean(history['l1'][:20]):.3f})")

policy.eval()
plain = LeRobotDataset("local/nb_demo", root=DEMO_ROOT, video_backend="pyav")
names = meta.features["action"]["names"][0]
j = names.index("kRightElbow")
truth = np.stack([plain[i]["action"].numpy() for i in range(int(plain.meta.episodes[0]["dataset_to_index"]))])
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(truth[:, j], color="0.6", lw=3, label="operador")
with torch.no_grad():
    for t in range(0, len(truth) - 48, 48):
        frame = plain[t]
        obs = preprocessor({k: frame[k] for k in ["observation.state", CAMERA]})
        chunk = postprocessor(policy.predict_action_chunk(obs))[0].cpu().numpy()
        ax.plot(range(t, t + 48), chunk[:, j])
ax.set_title("episodio visto en el entrenamiento · kRightElbow"); ax.legend(); plt.tight_layout()

Mirá bien el gráfico: cada bloque **arranca a la altura correcta**, pero sale
**casi plano**. Después de 600 pasos la red aprendió *dónde está* el brazo, no
*cómo se mueve*. Es lo primero que aprende un modelo que arranca de cero,
porque es lo más fácil: equivale a la referencia "quedarse quieto" del
notebook 05. El movimiento dentro del bloque llega con miles de pasos más.

Aun así la prueba sirve: la pérdida bajó a menos de un tercio, así que **el
pipeline aprende** (datos alineados, normalización bien, gradientes sanos). Si
hubiera quedado clavada, habría que buscar el bug antes de gastar horas de GPU.
Y no dice nada de si generaliza: para eso, muchos más pasos y episodios no
vistos (notebook 05).

### El comando de verdad

Lo mismo, con `lerobot-train`. 20 pasos, sólo para ver qué deja:

In [ ]:
run_dir = DATA_DIR / "runs" / "nb_l5_cli"
command = [
    str(Path(sys.executable).parent / "lerobot-train"),
    "--dataset.repo_id=local/nb_demo", f"--dataset.root={DEMO_ROOT}", "--dataset.video_backend=pyav",
    "--policy.type=act", f"--policy.device={DEVICE}", "--policy.push_to_hub=false",
    "--policy.chunk_size=48", "--policy.n_action_steps=48",
    "--policy.dim_model=192", "--policy.dim_feedforward=768", "--policy.n_heads=4",
    "--steps=20", "--batch_size=8", "--log_freq=5", "--save_freq=20", "--eval_freq=0",
    "--wandb.enable=false", "--num_workers=0", f"--output_dir={run_dir}",
]
subprocess.run(["rm", "-rf", str(run_dir)])
result = subprocess.run(command, capture_output=True, text=True)
print("\n".join(line for line in result.stderr.splitlines() if "step:" in line))

Cada línea de log trae: `smpl` (ejemplos vistos), `epch` (fracción del dataset
recorrida), `loss`, `grdn` (norma del gradiente), `lr`, y cuánto tardó la
actualización (`updt_s`) y la carga de datos (`data_s`). Si `data_s` se acerca
a `updt_s`, el cuello de botella es leer video, no la GPU.

Y lo que queda en disco:

In [ ]:
for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:7.2f} MB  {path.relative_to(run_dir)}")

- `checkpoints/000020/pretrained_model/`: **el modelo**, igual que el que nos
  pasaron de la primera ACT (pesos + configuración + procesadores). Es lo que
  se carga con `toolkit.policy.load`.
- `checkpoints/000020/training_state/`: el estado del optimizador y de los
  generadores aleatorios, para **retomar** un entrenamiento cortado
  (`--resume=true --config_path=.../train_config.json`).
- `checkpoints/last`: un atajo al último.

---

# Para profundizar

## 5. ¿Cuánto cuesta entrenar en serio?

La primera ACT: 30.000 pasos con lotes de 64. Una estimación gruesa a partir
de lo que medimos (asume que el tiempo crece proporcional al lote, que es
pesimista para una GPU):

In [ ]:
steps, batch = 30_000, 64
estimate_hours = seconds_per_step * (batch / 8) * steps / 3600
print(f"en esta Mac ({DEVICE}): ~{estimate_hours:.0f} h (estimación gruesa, no medida)")

Por eso el entrenamiento de verdad va a la VM con GPU NVIDIA o a la
workstation. El código y los comandos son los mismos: sólo cambia
`--policy.device=cuda`.

## 6. Las perillas que importan (y la disciplina)

| Perilla | Opción | Qué esperar |
|---|---|---|
| Aumentación de imágenes | `--dataset.image_transforms.enable=true` | Menos sobreajuste con pocos datos (L2) |
| Pasos | `--steps` | ACT suele mejorar en éxito "mucho después de que la pérdida se aplana" (README de ACT) |
| Bloque | `--policy.chunk_size`, `n_action_steps` | Suavidad contra reactividad (L3) |
| KL | `--policy.kl_weight` | Si baja, el VAE puede dejar de colapsar (L3) |
| Tamaño | `--policy.dim_model`, `dim_feedforward` | Capacidad contra costo y sobreajuste |
| Normalización | `--policy.normalization_mapping` | L4 |

Dos reglas de la casa (`AGENTS.md`):

- **Una variable por experimento**, con la métrica y el criterio declarados
  antes de entrenar.
- **Train y evaluación como datasets separados** (`split_dataset`, L2): en esta
  versión `--dataset.episodes` no filtra.

## Referencias

- `lerobot/scripts/lerobot_train.py` (dentro de `.venv`): el ciclo completo.
- `lerobot/datasets/factory.py`: cómo se arma el dataset, incluido el reemplazo
  por estadísticas de ImageNet.
- robomimic sobre elegir checkpoints: [robomimic.github.io/study](https://robomimic.github.io/study/).
- README de ACT: [github.com/tonyzhaozh/act](https://github.com/tonyzhaozh/act).